# import

### Picture for signal noise

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append("../General_Functions/")
from BasicData import *
from Flow_functions import * # import flow functions, Information to the agent
from Geometric_functions import * # Import rotation from euler angles
from Strategies_Evolution_SensoryNoise import * # Import strategies, Braitenberg or Triangulation
from Flow_functions import * # Import flow functions: stokeslet, grads, strains ... 

In [2]:
# Sign function
def calculate_S(n, v0, v1, v2):
    """
    Calculate S = (n·v2)||v0||^2 - (n·v0)(v0·v2)
    """
    n_dot_v2 = np.dot(n, v2)
    n_dot_v0 = np.dot(n, v0)
    v0_dot_v2 = np.dot(v0, v2)
    v0_norm_squared = np.dot(v0, v0)
    return n_dot_v2 * v0_norm_squared - n_dot_v0 * v0_dot_v2

In [5]:

def Evolution(Grad, Intensity, Duration):

    # Function to evaluate the scalar product between t at the end of the evolution
    t0_T, t1_T, t2_T = np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials))
    DotProduct_T = np.zeros((NPoints, NPoints, NTrials))
    # DotProduct_I = np.zeros((NPoints, NPoints, NTrials))

    for k in range(NPoints):
        for i in range(NPoints):
            # Position the agent 
            AgentPosition =  np.array([z[k], x[i], 0.])

            # Source position
            OptimalDirection = -AgentPosition/np.linalg.norm(AgentPosition)

            for trials_i in range(NTrials):
                # Random orientation of agent axis
                alpha, beta, gamma = (2 * np.random.rand(3) - 1) * np.pi
                R = Rotation(alpha, np.abs(beta), gamma)
                n_T = np.dot(R,[0, 1, 0])
                b_T = np.dot(R,[0, 0, 1])
                t_T = np.dot(R,[1, 0, 0])
                
                
                for time in range(Duration):
                    n_T, b_T, t_T = Evolution_Triangulation_NoiseSign(n_T, b_T, t_T, AntennaeLength, AgentPosition, Grad, e3, dt, roll, Intensity)
                 
                t0_T[k, i, trials_i], t1_T[k, i, trials_i], t2_T[k, i, trials_i] = t_T

                # Dot product Evaluation
                DotProduct_T[k, i, trials_i] = np.dot(t_T, OptimalDirection)


    return DotProduct_T, \
        t0_T, t1_T

In [6]:
# Data
NPoints = 80 # Grid number of points per axis
NTrials = 30 # Number of trial x point

Duration = 255
AntennaeLength = .01 # Sensors distance, on antennae
x = np.linspace(0.1, 5, NPoints) # x axis
z = np.linspace(0.1, 5, NPoints) # z axis
X, Z = np.meshgrid(x, z)
e3 = np.array([1, 0, 0]) # Source flow axis of symmetry, scheme zxy
dt = .1 # Timesteps 
roll = .4 # Roll Intensity
NameStrategies = ["T"] #_not", "T", "I"] 

In [4]:
# 

# Information flow 
GradName = ['GradientStresslet',  'StrainStresslet', 'StrainStokeslet', 'GradientStokeslet'] # 'ShearQuadruplet', 'GradientQuadruplet', 
Grads =  [GradientStresslet, StrainStresslet, StrainStokeslet, GradientStokeslet] # ShearQuadruplet, GradientQuadruplet



for grads_i, Grad in enumerate(Grads):
    
    Intensity = 0.1*np.linalg.norm(Grad(e3, np.array([2,0,0])))
    # t axis for Triangulation strategy absence of roll
    t0_T, t1_T = np.zeros((NPoints, NPoints, NTrials)), np.zeros((NPoints, NPoints, NTrials))

    # Evaluate the strategy 2D map, vector P 
    Avg_P_T = np.zeros((NPoints, NPoints, NTrials))

    # Call Evaluation function and evaluate 
    Avg_P_T, t0_T, t1_T = Evolution(Grad, Intensity, Duration)
    
    # Save data

    AllAvg = [Avg_P_T] #, Avg_P_T, Avg_P_I]
    Allt0 = [t0_T] # _not, t0_T, t0_I]
    Allt1 = [t1_T] # _not, t1_T, t1_I]

    for i, strat_name in enumerate(NameStrategies):
        np.savetxt("Data_SensoryNoise/"+str(GradName[grads_i])+"_Avg_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".txt", np.mean(AllAvg[i], axis = 2), delimiter = ',')
        np.savetxt("Data_SensoryNoise/"+str(GradName[grads_i])+"_t0_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".txt", np.mean(Allt0[i], axis = 2), delimiter = ',')
        np.savetxt("Data_SensoryNoise/"+str(GradName[grads_i])+"_t1_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".txt", np.mean(Allt1[i], axis = 2), delimiter = ',')

#plt.show()


NameError: name 'Evolution' is not defined

In [7]:
# Plot data from Save 
# Information flow 
GradName = ['GradientStresslet',  'StrainStresslet', 'StrainStokeslet', 'GradientStokeslet'] # 'ShearQuadruplet', 'GradientQuadruplet', 
Grads =  [GradientStresslet, StrainStresslet, StrainStokeslet, GradientStokeslet] # ShearQuadruplet, GradientQuadruplet

import scipy.ndimage as ndimage # import to smooth plots
from scipy.special import erf
Trial = 10
for grads_i, Grad in enumerate(GradName):
    Intensity = 0.1*np.linalg.norm(Grads[grads_i](e3, np.array([2,0,0]))) # This is noise intensity
    for i, strat_name in enumerate(NameStrategies):
        # Compute GradField Intensity over X, Z
        GradField = np.zeros_like(X)
        gradient_values = np.zeros(Trial)
        for ix in range(X.shape[0]):
            for iz in range(X.shape[1]):
                for t in range(Trial):
                    theta = 2 * np.pi * t / Trial  # Angle for this trial
                    n_reference = np.array([np.cos(theta), np.sin(theta), 0.0])  # Unit vector on a circle
                    position = np.array([Z[ix, iz], X[ix, iz], 0.0]) 
                    gradient_values = np.linalg.norm(np.dot(n_reference, Grads[grads_i](e3, position)))
                GradField[ix, iz] = np.mean(gradient_values)
        # Extract data from saved file
        with open("Data_SensoryNoise/"+str(GradName[grads_i])+"_Avg_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".txt", 'r') as the_file:
            Avg = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 

        with open("Data_SensoryNoise/"+str(GradName[grads_i])+"_t0_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".txt", 'r') as the_file:

            t0 = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 
        with open("Data_SensoryNoise/"+str(GradName[grads_i])+"_t1_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".txt", 'r') as the_file:

            t1 = np.array([np.asarray(each_line.split(",")).astype(float) for each_line in the_file]) 
        
        # Plot parameters
        # Define SENSORY_PERIMETER
        Sensitivity = 0.2
        SENSORY_PERIMETER = AntennaeLength * (Sensitivity / Intensity)**(1/4)
        threshold_value = (0.2/Intensity)**(1/4)
        CMAP = 'RdBu_r' #'spring'
        STREAMCOLOR = 'black'
        MIN_VALUE = -1
        MAX_VALUE = 1
        fig1, axs = plt.subplots(nrows=1, ncols=1, figsize=(8,8), sharex='col', sharey='row')
        ax1 = axs
        X, Z = np.meshgrid(x, z, indexing='xy')

        # Add data to image grid
        levels = np.linspace(-1, 1.00001, 14)

        Avg = ndimage.gaussian_filter(Avg, sigma=1.0, order=0)
        im = ax1.contourf(X, Z, Avg, cmap = CMAP, levels = levels, vmin= MIN_VALUE, vmax = MAX_VALUE, alpha = 0.5)
        ax1.streamplot(X, Z, t1, t0, color = STREAMCOLOR,  density = .3,  arrowsize = 2.5, broken_streamlines=False)# Plot contour line at SENSORY_PERIMETER
        contour_line = ax1.contour(X, Z, GradField, levels=[threshold_value], colors='green', linewidths=3, linestyles = '--')
        #ax1.clabel(contour_line, fmt={SENSORY_PERIMETER: f'{SENSORY_PERIMETER:.2f}'}, inline=True, fontsize=10)

        #PlotCircles(ax1, .5)
        ax1.set_xlim([.1, 2.5])
        ax1.set_ylim([.1, 2.5])
        ax1.set_xticks([])
        ax1.set_yticks([])



        #plt.colorbar(im, ax = axs, ticks = np.arange(-1, 1.1, 0.5), location= 'bottom')
        # Uncomment to save
        #plt.savefig("Plots_SensoryNoise/"+str(GradName[grads_i])+"_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".eps", format = 'eps', bbox_inches='tight')
        #plt.savefig("Plots_SensoryNoise/"+str(GradName[grads_i])+"_"+str(strat_name)+"_Intensiy_"+str(np.round(Intensity, 3))+".pdf", bbox_inches='tight')
        plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'Data_SensoryNoise/GradientStresslet_Avg_T_Intensiy_0.061.txt'

In [72]:
Avg

array([[ 0.77487416,  0.75917879,  0.75030984, ..., -0.01005116,
        -0.02630265, -0.06248282],
       [ 0.78788029,  0.77382052,  0.75930519, ..., -0.00548107,
        -0.02696871, -0.05626124],
       [ 0.80204164,  0.79229675,  0.77671784, ...,  0.00143917,
        -0.01503231, -0.03461683],
       ...,
       [-0.04055096, -0.02391996, -0.0111635 , ...,  0.01418187,
         0.02772884,  0.02796445],
       [-0.01192699,  0.00562306,  0.01422948, ...,  0.03964437,
         0.04770647,  0.04033279],
       [ 0.00607341,  0.0224846 ,  0.02402867, ...,  0.05144242,
         0.04967912,  0.03096043]])